# M5 seed 43·44·45 체크포인트 진단 — 재학습 없음
저장된 선택 checkpoint에서 BPR/L2 gradient와 N/V 점수차를 분해합니다. optimizer·파라미터 갱신·test/holdout 평가가 없습니다.
동일 probe seed로 시드당 학습행 8192개 × 3배치를 추출합니다(배치 간 중복 가능). K=1 균등 미관측 음성과 양성 상품 leave-one-out을 유지합니다. 전체 Top-K 평가가 아니라 **학습 표본의 양성–음성 점수차 진단**입니다.
큰 L2/BPR 비율만으로 과규제를 확정하지 마세요. 방향(cosine), 표본별 변동, 실제 규제 변경 실험이 필요합니다. q−0.5는 고정 가중치에서의 점수 진단이지 재학습 대조군이 아닙니다. bias도 상품별 점수를 바꾸므로 자동으로 무의미한 항이 아닙니다.
Drive의 기존 결과 JSON과 선택 checkpoint가 모두 필요합니다. 파일 hash·모델 소스·입력 hash·epoch가 다르면 중단하고 학습으로 대체하지 않습니다. 기존 결과는 읽기만 하며 별도 진단 폴더에 저장합니다.
별도 GPU 런타임 권장. 데이터 준비와 9개 진단 배치 비용이 들며 100/300 epoch 학습은 하지 않습니다. 실제 소요시간은 아직 측정하지 않았습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,subprocess,json
SOURCE_COMMIT='e4d903096299d0e07164658fdc1bca954d53647d'
REPO=Path('/content/clv-nv-diagnostic-'+SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent==REPO.resolve(), '별도 런타임 또는 세션 재시작 필요'
os.chdir(REPO);sys.path.insert(0,str(REPO))
import torch,pandas as pd
import clv_linear_nv_checkpoint_diagnostic as diagnostic
assert torch.cuda.is_available(), 'GPU 런타임 권장: CPU에서 실수로 실행하지 않도록 차단합니다.'


In [ ]:
ROOT=Path('/content/drive/MyDrive/논문/data')
REPORTS=[
 ROOT/'results_v3_dunnhumby_history_linear_nv_es_v2/reports/30287cad8e5ed1d1/result.json',
 ROOT/'results_v3_dunnhumby_history_linear_nv_es_seed44_v2/reports/2e4275a50b8a5de1/result.json',
 ROOT/'results_v3_dunnhumby_history_linear_nv_es_seed45_v2/reports/2e97778737396d2e/result.json']
for path in REPORTS:
    assert path.is_file(), f'원본 결과가 없습니다: {path}'
    report=json.loads(path.read_text())
    arm=next(a for a in report['arms'] if a['model_id']=='m5_linear_nv')
    print(arm['seed'], arm['selected_epoch'], arm['checkpoint'], Path(arm['checkpoint']).is_file())
OUT=ROOT/'diagnostics_linear_nv_checkpoint_43_44_45_v1'
print('학습 없음. 진단 표본: 시드당 3배치 × 8192행. 출력:',OUT)


In [ ]:
paths=diagnostic.run(REPORTS,OUT,batches=3,batch_size=8192)
g=pd.read_csv(paths['gradients']);s=pd.read_csv(paths['score_components'])
print(g.to_string(index=False))
print(s.to_string(index=False))
print('코사인 음수: BPR과 규제가 반대 방향. 이것만으로 실패 원인을 확정하지 않습니다.')
from zipfile import ZipFile,ZIP_DEFLATED
from google.colab import files
archive=Path('/content/linear_nv_checkpoint_diagnostic.zip')
with ZipFile(archive,'w',compression=ZIP_DEFLATED) as z:
    for path in paths.values(): z.write(path,arcname=Path(path).name)
files.download(str(archive))
